In [19]:
%load_ext autoreload
%autoreload 2

import sys
import os
import time
from collections import defaultdict

# Add parent directory to path so we can import embeddings
sys.path.insert(0, os.path.abspath(".."))

import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from embeddings.model import EmbeddingModel
from embeddings.dataset import load_snli_clean
from embeddings.loss import TripletLoss

In [20]:
# Hyperparameters
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
EPOCHS = 3
MARGIN = 0.1
MODEL_NAME = "distilbert-base-uncased"
TRAIN_SUBSET = 20000
TIME_LIMIT_SECONDS = 10 * 60  # cap total training at 10 minutes
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Save path: <project_root>/saved/model/trained_embedder.pt
PROJECT_ROOT = os.path.abspath("..")
SAVE_PATH = os.path.join(PROJECT_ROOT, "saved", "model", "trained_embedder.pt")

print(f"Using device: {DEVICE}")
print(f"Batch size: {BATCH_SIZE}, Learning rate: {LEARNING_RATE}, Epochs: {EPOCHS}")
print(f"Time limit: {TIME_LIMIT_SECONDS}s")
print(f"Save path: {SAVE_PATH}")

Using device: cpu
Batch size: 32, Learning rate: 2e-05, Epochs: 3
Time limit: 600s
Save path: /Users/arvindranganathraghuraman/Desktop/Personal_Projects/DistilQA/DistilQA/saved/model/trained_embedder.pt


In [21]:
# Load and filter SNLI dataset
print("Loading SNLI dataset...")
train_dataset = load_snli_clean("train").shuffle(seed=42).select(range(TRAIN_SUBSET))
print(f"Using subset of {len(train_dataset)} training examples")

# Initialize embedding model
print(f"Loading model: {MODEL_NAME}...")
model = EmbeddingModel(MODEL_NAME, device=DEVICE)
print("Model loaded")

Loading SNLI dataset...
Using subset of 20000 training examples
Loading model: distilbert-base-uncased...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 14138.42it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded


In [22]:
# Loss and optimizer
criterion = TripletLoss(margin=MARGIN)
optimizer = optim.Adam(model.model.parameters(), lr=LEARNING_RATE)

print(f"Loss: TripletLoss (margin={MARGIN})")
print(f"Optimizer: Adam (lr={LEARNING_RATE})")

Loss: TripletLoss (margin=0.1)
Optimizer: Adam (lr=2e-05)


In [23]:
# Build SNLI triplets: group by premise, keep groups that have both
# entailment (label=0) and contradiction (label=2). Each triplet =
# (anchor=premise, positive=entailment_hyp, negative=contradiction_hyp).
class SNLITripletDataset(Dataset):
    def __init__(self, hf_dataset):
        groups = defaultdict(lambda: {"pos": [], "neg": []})
        for ex in hf_dataset:
            if ex["label"] == 0:
                groups[ex["premise"]]["pos"].append(ex["hypothesis"])
            elif ex["label"] == 2:
                groups[ex["premise"]]["neg"].append(ex["hypothesis"])

        self.triplets = []
        for premise, hyps in groups.items():
            if hyps["pos"] and hyps["neg"]:
                # take all combinations (usually 1 of each in SNLI)
                for p in hyps["pos"]:
                    for n in hyps["neg"]:
                        self.triplets.append((premise, p, n))

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        anchor, positive, negative = self.triplets[idx]
        return {"anchor": anchor, "positive": positive, "negative": negative}


train_triplets = SNLITripletDataset(train_dataset)
train_loader = DataLoader(train_triplets, batch_size=BATCH_SIZE, shuffle=True)
print(f"Built {len(train_triplets)} triplets from {len(train_dataset)} SNLI rows")

Built 452 triplets from 20000 SNLI rows


In [24]:
# Training loop with hard time cap (TIME_LIMIT_SECONDS).
# We stop mid-epoch the moment the budget is exceeded, so weights are
# always saved even on slow CPU training.
start = time.time()
stopped_early = False

for epoch in range(1, EPOCHS + 1):
    if stopped_early:
        break
    model.model.train()
    epoch_loss = 0.0
    num_batches = 0
    for batch in train_loader:
        elapsed = time.time() - start
        if elapsed >= TIME_LIMIT_SECONDS:
            print(f"Time budget reached ({elapsed:.0f}s) — stopping early")
            stopped_early = True
            break

        anchor_emb = model.get_embedding(list(batch["anchor"]))
        positive_emb = model.get_embedding(list(batch["positive"]))
        negative_emb = model.get_embedding(list(batch["negative"]))

        loss = criterion(anchor_emb, positive_emb, negative_emb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

    if num_batches > 0:
        avg_loss = epoch_loss / num_batches
        tag = " (partial)" if stopped_early else ""
        print(f"Epoch {epoch}/{EPOCHS} - avg loss: {avg_loss:.4f} - batches: {num_batches}{tag}")

print(f"Training complete in {time.time() - start:.0f}s")

Epoch 1/3 - avg loss: 0.0569 - batches: 15
Epoch 2/3 - avg loss: 0.0248 - batches: 15
Epoch 3/3 - avg loss: 0.0141 - batches: 15
Training complete in 67s


In [26]:
# Persist trained weights (inline torch.save — avoids depending on
# methods that may not exist on a stale model instance)
os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
torch.save(model.model.state_dict(), SAVE_PATH)
print(f"Saved weights to {SAVE_PATH}")

Saved weights to /Users/arvindranganathraghuraman/Desktop/Personal_Projects/DistilQA/DistilQA/saved/model/trained_embedder.pt
